In [30]:
from pathlib import Path
import polars as pl
import re
# import geopandas as gpd
# from shapely.geometry import shape

In [9]:
raw_dir = Path("data/raw")
bronze_dir = Path("data/bronze/buildings")
bronze_dir.mkdir(parents=True, exist_ok=True)

pattern = re.compile(
    r'(?P<country>.+)_(?P<quadkey>\d+)_(?P<upload_date>\d{4}-\d{2}-\d{2})\.csv\.gz$'
)

for path in raw_dir.glob("*.csv.gz"):
    match = pattern.match(path.name)
    if not match:
        print(f"Skipping unexpected filename: {path.name}")
        continue

    meta = match.groupdict()

    df = (
        pl.read_ndjson(path)
        .unnest("properties")   # turns properties.height / confidence into columns
        .with_columns(
            pl.lit(meta["country"]).alias("country"),
            pl.lit(meta["quadkey"]).alias("quadkey"),
            pl.lit(meta["upload_date"]).alias("upload_date"),
            pl.lit(path.name).alias("source_file"),
        )
    )

    out_path = bronze_dir / path.name.replace(".csv.gz", ".parquet")
    df.write_parquet(out_path)

    print(f"Wrote: {out_path}")

data/raw/Indonesia_132223000_2026-02-23.csv.gz
data/raw/Indonesia_132223020_2026-02-23.csv.gz
data/raw/Indonesia_132223030_2026-02-23.csv.gz
data/raw/Indonesia_132223022_2026-02-23.csv.gz
data/raw/Indonesia_132223010_2026-02-23.csv.gz
data/raw/Indonesia_132222113_2026-02-23.csv.gz
data/raw/Indonesia_132223013_2026-02-23.csv.gz
data/raw/Indonesia_132220333_2026-02-23.csv.gz
data/raw/Indonesia_132223001_2026-02-23.csv.gz
data/raw/Indonesia_132223021_2026-02-23.csv.gz
data/raw/Indonesia_132222111_2026-02-23.csv.gz
data/raw/Indonesia_132223011_2026-02-23.csv.gz
data/raw/Indonesia_132223012_2026-02-23.csv.gz
data/raw/Indonesia_132223002_2026-02-23.csv.gz
data/raw/Indonesia_132223003_2026-02-23.csv.gz
data/raw/Indonesia_132223023_2026-02-23.csv.gz
data/raw/Indonesia_132223031_2026-02-23.csv.gz


In [50]:
# #See buildings
# df = pl.read_ndjson("data/raw/Indonesia_132223122_2026-02-23.csv.gz").head(100)
# geoms = [shape(g) for g in df["geometry"].to_list()]
#
# gdf = gpd.GeoDataFrame(
#     {"row_id": range(len(geoms))},
#     geometry=geoms,
#     crs="EPSG:4326"
# )
# gdf.explore()